# 04a — Exposure & Panel Diagnostic Checks

Validates the panel constructed in NB04 before any analysis. Three questions:

1. **Differential attrition** — do exposed and unexposed users drop out of the panel at different rates?
2. **Pre-period quality** — how much baseline data do exposed users actually have, and is it systematically thin?
3. **Anchor comment timing** — when in the Sep–Nov window are users actually getting exposed?

These checks inform whether the panel is fit for causal analysis and flag any design limitations to address in the paper.

**Inputs:** `exposure_labels_v2.parquet` (NB03), `panel_scores_v2.parquet` (NB04), `post_level_scores_v2.parquet` (NB04), `anchor_posts_v2.parquet` (NB03), raw comments JSONL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed_v2' / 'gradadmissions'
RAW_DIR  = ROOT
FIG_DIR  = ROOT / 'figures'

labels   = pd.read_parquet(DATA_DIR / 'exposure_labels_v2.parquet')
panel    = pd.read_parquet(DATA_DIR / 'panel_scores_v2.parquet')
post_lvl = pd.read_parquet(DATA_DIR / 'post_level_scores_v2.parquet')
anchors  = pd.read_parquet(DATA_DIR / 'anchor_posts_v2.parquet')

print(f'Exposure labels: {len(labels):,} rows  |  Exposed: {labels["exposed"].sum():,}  |  Unexposed: {(~labels["exposed"]).sum():,}')
print(f'Panel:           {len(panel):,} rows  |  Unique users: {panel["author"].nunique():,}')

## 1) Differential attrition

Of the users labeled in NB03, what fraction made it into the panel (i.e., had activity in both pre AND post windows)? If exposed and unexposed users drop out at similar rates, attrition is not biasing the comparison.

In [ ]:
panel_keys = set(zip(panel['author'], panel['cycle']))
labels['in_panel'] = labels.apply(lambda r: (r['author'], r['cycle']) in panel_keys, axis=1)

attrition = (
    labels.groupby('exposed')['in_panel']
          .agg(in_panel='sum', total='count')
          .assign(retention_pct=lambda x: (x['in_panel'] / x['total'] * 100).round(1),
                  dropout_pct=lambda x: (100 - x['in_panel'] / x['total'] * 100).round(1))
)
attrition.index = ['Unexposed', 'Exposed']
print('Panel retention by exposure status:')
display(attrition)

diff = attrition.loc['Exposed','retention_pct'] - attrition.loc['Unexposed','retention_pct']
print(f'\nRetention difference (Exposed - Unexposed): {diff:+.1f} pp')
if abs(diff) < 3:
    print('=> No meaningful differential attrition. Both groups drop out at equivalent rates.')
else:
    print('=> Differential attrition detected — investigate further.')

In [ ]:
# Also check by cycle
print('Retention by exposure status and cycle:')
by_cycle = (
    labels.groupby(['cycle','exposed'])['in_panel']
          .agg(in_panel='sum', total='count')
          .assign(retention_pct=lambda x: (x['in_panel'] / x['total'] * 100).round(1))
)
display(by_cycle)

## 2) Pre-period quality for exposed users

The pre-period for exposed users is truncated at their first anchor comment. Users who commented early in Sep have almost no pre-period; November commenters have a much fuller baseline. This section documents how thin those baselines actually are.

In [ ]:
pre = post_lvl[post_lvl['window'] == 'pre'].copy()
pre['created_dt'] = pd.to_datetime(pre['created_dt'])

span = (
    pre.groupby(['author','cycle'])
       .agg(pre_first=('created_dt','min'),
            pre_last=('created_dt','max'),
            pre_n_posts=('mean_mh_score','count'))
       .reset_index()
)
span['pre_span_days'] = (span['pre_last'] - span['pre_first']).dt.days

panel_with_span = panel.merge(span, on=['author','cycle'], how='left')
exposed_panel   = panel_with_span[panel_with_span['exposed']]
unexposed_panel = panel_with_span[~panel_with_span['exposed']]

print('Pre-period span (days) — EXPOSED users in panel:')
print(exposed_panel['pre_span_days'].describe().round(1))
print()
print('Pre-period span (days) — UNEXPOSED users in panel:')
print(unexposed_panel['pre_span_days'].describe().round(1))

In [ ]:
print('Exposed users by pre-period span threshold:')
for threshold in [0, 3, 7, 14, 30]:
    n = (exposed_panel['pre_span_days'] <= threshold).sum()
    pct = n / len(exposed_panel) * 100
    label = f'== {threshold}' if threshold == 0 else f'<= {threshold}'
    print(f'  Span {label:6} days: {n:4d} / {len(exposed_panel):,} ({pct:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(exposed_panel['pre_span_days'].dropna(), bins=30, color='firebrick', alpha=0.7, edgecolor='white')
axes[0].axvline(7,  color='black', linestyle='--', linewidth=1, label='7 days')
axes[0].axvline(14, color='gray',  linestyle='--', linewidth=1, label='14 days')
axes[0].set_xlabel('Pre-period span (days)')
axes[0].set_ylabel('Number of exposed users')
axes[0].set_title('Distribution of pre-period span\n(exposed users in panel)')
axes[0].legend()

# Pre-period span by month of first activity
exposed_panel2 = exposed_panel.copy()
exposed_panel2['pre_first_month'] = exposed_panel2['pre_first'].dt.to_period('M').astype(str)
month_span = exposed_panel2.groupby('pre_first_month')['pre_span_days'].median().reset_index()

axes[1].bar(month_span['pre_first_month'], month_span['pre_span_days'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Month of first pre-period activity')
axes[1].set_ylabel('Median pre-period span (days)')
axes[1].set_title('Median pre-period span by month\n(confirms late-commenter baseline collapse)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_preperiod_span.png', dpi=150, bbox_inches='tight')
plt.show()

## 3) Anchor comment timing

When in the Sep–Nov window are users actually getting exposed? This determines how much pre-period data is available before treatment for the typical exposed user.

In [ ]:
anchor_ids    = set(anchors['id'])
exposed_users = set(labels[labels['exposed']]['author'])

rows = []
with open(RAW_DIR / 'Grad Admissions Comments.jsonl') as f:
    for line in f:
        obj = json.loads(line)
        if obj.get('link_id','').replace('t3_','') in anchor_ids:
            author = obj.get('author','')
            if author in exposed_users:
                rows.append({
                    'author':     author,
                    'created_dt': pd.to_datetime(obj['created_utc'], unit='s', utc=True),
                })

anchor_comments = pd.DataFrame(rows)
anchor_comments['month'] = anchor_comments['created_dt'].dt.to_period('M')

# First anchor comment per user (= moment of exposure)
first_comment = (
    anchor_comments.sort_values('created_dt')
                   .groupby('author')['created_dt'].first()
                   .reset_index()
)
first_comment['month'] = first_comment['created_dt'].dt.to_period('M')

print('All anchor comments by month:')
print(anchor_comments.groupby('month').size().to_string())
print()
print('First anchor comment (exposure moment) by month:')
print(first_comment.groupby('month').size().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# All anchor comments
all_by_month = anchor_comments.groupby('month').size().reset_index(name='n')
all_by_month['month_str'] = all_by_month['month'].astype(str)
axes[0].bar(all_by_month['month_str'], all_by_month['n'], color='steelblue', alpha=0.8)
axes[0].set_title('All anchor comments by month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Comment count')
axes[0].tick_params(axis='x', rotation=45)

# First anchor comment per user
first_by_month = first_comment.groupby('month').size().reset_index(name='n')
first_by_month['month_str'] = first_by_month['month'].astype(str)
axes[1].bar(first_by_month['month_str'], first_by_month['n'], color='firebrick', alpha=0.8)
axes[1].set_title('First anchor comment per user (exposure moment)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of users')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_anchor_comment_timing.png', dpi=150, bbox_inches='tight')
plt.show()

# Sep/Oct/Nov breakdown within anchor window
sep_nov = first_comment[first_comment['month'].astype(str).str.contains('2023-09|2023-10|2023-11|2024-09|2024-10|2024-11')]
print('Exposure timing within Sep-Nov window:')
print(sep_nov['month'].value_counts().sort_index().to_string())
total = len(sep_nov)
nov   = sep_nov[sep_nov['month'].astype(str).str.endswith('-11')]
print(f'\nNovember share: {len(nov):,} / {total:,} = {len(nov)/total*100:.1f}% of within-window exposures')

## 4) Baseline characteristic comparison: panel vs. dropped users

Among exposed users who were labeled in NB03, are those who made it into the panel systematically different from those who dropped out? If panel users differ on pre-period scores or post counts, the panel is a selected sample.

In [ ]:
# Panel users: have pre_mh_score from NB04
panel_exposed = panel[panel['exposed']][['author','cycle','pre_mh_score','pre_n_posts','post_n_posts']].copy()
panel_exposed['in_panel'] = True

# All exposed from labels
all_exposed = labels[labels['exposed']][['author','cycle']].copy()
all_exposed = all_exposed.merge(panel_exposed, on=['author','cycle'], how='left')
all_exposed['in_panel'] = all_exposed['in_panel'].fillna(False)

# Post-period activity for dropped users (they have post activity but no pre)
post_scores = (
    post_lvl[post_lvl['window']=='post']
    .groupby(['author','cycle'])
    .agg(post_n_posts_raw=('mean_mh_score','count'),
         post_mh_score_raw=('mean_mh_score','mean'))
    .reset_index()
)
all_exposed = all_exposed.merge(post_scores, on=['author','cycle'], how='left')

print('Exposed users: panel vs. dropped out')
print(all_exposed.groupby('in_panel')[['pre_n_posts','post_n_posts_raw','post_mh_score_raw']].agg(['mean','median']).round(3))

## 5) Summary of findings

Run this cell after all checks to print a concise diagnostic summary.

In [ ]:
exp_ret   = labels[labels['exposed']]['in_panel'].mean() * 100
unexp_ret = labels[~labels['exposed']]['in_panel'].mean() * 100
span0     = (exposed_panel['pre_span_days'] == 0).mean() * 100
span7     = (exposed_panel['pre_span_days'] < 7).mean() * 100
nov_share = len(nov) / total * 100

print('=' * 60)
print('PANEL DIAGNOSTIC SUMMARY')
print('=' * 60)
print(f'\n[1] DIFFERENTIAL ATTRITION')
print(f'    Exposed retention:   {exp_ret:.1f}%')
print(f'    Unexposed retention: {unexp_ret:.1f}%')
print(f'    Difference:          {exp_ret - unexp_ret:+.1f} pp  => {"CLEAN" if abs(exp_ret-unexp_ret) < 3 else "FLAG"}')

print(f'\n[2] PRE-PERIOD QUALITY (exposed panel users)')
print(f'    Single-day baseline (span=0):  {span0:.1f}%')
print(f'    Thin baseline (span < 7 days): {span7:.1f}%')
print(f'    Median span: {exposed_panel["pre_span_days"].median():.0f} days')

print(f'\n[3] EXPOSURE TIMING (within Sep-Nov window)')
print(f'    November share of exposures: {nov_share:.1f}%')
print(f'    => November-heavy exposure = majority of users have thin pre-periods')

print(f'\n[IMPLICATION FOR PAPER]')
print(f'    Attrition is non-differential: safe to proceed.')
print(f'    Pre-period thinness is a real limitation: report span distribution,')
print(f'    cite NB06a sensitivity check (pre >= 7 days) as robustness evidence.')
print('=' * 60)